# Task 4: Two from-scratch DPO runs

**Settings:** Accelerator **GPU T4 x2**, Internet **on**.
**Input:** Add Input > Your Work > the saved output of **pa2-03-sft** (the SFT checkpoint).
Run with **Save Version > Save & Run All (Commit)**.

Both runs start from the SFT best checkpoint. The trainable policy sits on GPU 0; a frozen copy of
the same checkpoint (the reference) sits on GPU 1 and is never updated. Loop: `src/pa2/train_dpo.py`.

1. **HH-RLHF** (`configs/dpo_hh.yaml`): 15,000 single-turn pairs, half harmless-base and half helpful-base.
2. **UltraFeedback** (`configs/dpo_ultrafeedback.yaml`): 15,000 non-tied pairs.

For each run: data preparation, an 8-step timing run (checks that step 0 gives loss ln 2 and margin 0,
memory and projected time), the full run with validation every 50 steps, and the curves.
Checkpoints stay in this notebook's output (`outputs/dpo/<run>/run/best` and `final`) for Task 6.
The small files are archived as `task4_dpo_outputs.zip`.

In [ ]:
REPO_URL = "https://github.com/mabdullahi7780/small-model-alignment-starter.git"
BRANCH = "main"
WORK_DIR = "/kaggle/working"
SCRATCH_DIR = "/tmp"
REPO_DIR = f"{SCRATCH_DIR}/small-model-alignment-starter"
OUT_DIR = f"{WORK_DIR}/outputs/dpo"
RUNS = {"hh": ("configs/dpo_hh.yaml", "DPO on HH-RLHF"),
        "ultrafeedback": ("configs/dpo_ultrafeedback.yaml", "DPO on UltraFeedback")}
SMOKE_STEPS = 8
MAX_PEAK_GIB = 13.0          # policy GPU, allocated by PyTorch
MAX_HOURS_PER_RUN = 3.0
RUN_SETTINGS = {"runs": RUNS, "smoke_steps": SMOKE_STEPS, "max_peak_gib": MAX_PEAK_GIB, "max_hours_per_run": MAX_HOURS_PER_RUN}

In [ ]:
import os
import subprocess
import sys


def run(cmd, cwd=None, env=None, check=True):
    print("$", " ".join(cmd), flush=True)
    return subprocess.run(cmd, cwd=cwd, env=env, check=check)


if not os.path.isdir(REPO_DIR):
    run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR])
run(["git", "log", "-1", "--format=commit %H%nauthor date %ad%nsubject %s"], cwd=REPO_DIR)

In [ ]:
# Kaggle's preinstalled torch is built for its GPU runtime; pin it so pip can never replace it.
TORCH_BEFORE = subprocess.run(
    [sys.executable, "-c", "import torch; print(torch.__version__)"],
    capture_output=True, text=True, check=True,
).stdout.strip()
print("Preinstalled torch:", TORCH_BEFORE)

constraint_file = f"{SCRATCH_DIR}/torch-constraint.txt"
with open(constraint_file, "w") as handle:
    handle.write(f"torch=={TORCH_BEFORE}\n")

run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt", "-c", constraint_file], cwd=REPO_DIR)
run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"], cwd=REPO_DIR)

In [ ]:
import json

COMMIT = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, capture_output=True, text=True, check=True).stdout.strip()
os.makedirs(f"{OUT_DIR}/logs", exist_ok=True)
with open(f"{OUT_DIR}/run_info.json", "w") as handle:
    json.dump({"commit": COMMIT, "torch": TORCH_BEFORE, **RUN_SETTINGS}, handle, indent=2)
print("commit", COMMIT)

In [ ]:
def run_on_gpus(jobs, tag):
    """Start one subprocess per job, each pinned to its own GPU, and wait for all of them.

    jobs: list of (name, gpu_index, argv). Each job's output goes to logs/<tag>_<name>.log.
    """
    # expandable_segments lets the CUDA allocator reuse freed memory instead of fragmenting it.
    env_base = {**os.environ, "PYTHONPATH": f"{REPO_DIR}/src:{REPO_DIR}", "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"}
    running = []
    for name, gpu, argv in jobs:
        log_path = f"{OUT_DIR}/logs/{tag}_{name}.log"
        handle = open(log_path, "w")
        print(f"start {tag}/{name} on GPU {gpu}", flush=True)
        process = subprocess.Popen(
            [sys.executable, *argv], cwd=REPO_DIR, stdout=handle, stderr=subprocess.STDOUT,
            env={**env_base, "CUDA_VISIBLE_DEVICES": str(gpu)},
        )
        running.append((name, process, handle, log_path))
    failed = []
    for name, process, handle, log_path in running:
        code = process.wait()
        handle.close()
        print(f"\n===== {tag}/{name}: exit code {code} =====")
        print(open(log_path).read()[-8000:])
        if code:
            failed.append(name)
    if failed:
        raise RuntimeError(f"{tag}: failed jobs {failed}; see {OUT_DIR}/logs")

## The SFT checkpoint (policy start and frozen reference)

In [ ]:
import glob

# The SFT notebook's saved output, attached with Add Input > Your Work > pa2-03-sft.
found = sorted(glob.glob("/kaggle/input/**/outputs/sft/run/best/model.safetensors", recursive=True))
if not found:
    raise RuntimeError("SFT checkpoint not found: attach the output of the pa2-03-sft notebook as an input")
SFT_MODEL = os.path.dirname(found[0])
print("SFT checkpoint:", SFT_MODEL)
print(open(f"{SFT_MODEL}/checkpoint_info.json").read())

In [ ]:
import math


def dpo_run(name, config_file, title):
    """Data prep, a timing run with checks, the full run and its plot. Skips work already done."""
    run_dir = f"{OUT_DIR}/{name}"
    summary_path = f"{run_dir}/run/summary.json"
    if os.path.exists(summary_path):
        done = json.load(open(summary_path))
        if done["steps_run"] == done["total_steps"]:
            print(f"{name}: already complete, skipping")
            return done

    if not os.path.exists(f"{run_dir}/data/train.jsonl"):
        run_on_gpus([("prep", "", ["-m", "pa2.dpo_data", "--config", config_file, "--out-dir", f"{run_dir}/data"])], f"{name}_data")
    print(open(f"{run_dir}/data/stats.json").read())

    common = ["-m", "pa2.train_dpo", "--config", config_file, "--sft-model", SFT_MODEL, "--data-dir", f"{run_dir}/data"]
    run_on_gpus([("smoke", "0,1", [*common, "--out-dir", f"{run_dir}/smoke", "--max-steps", str(SMOKE_STEPS), "--no-save"])], f"{name}_smoke")
    smoke = json.load(open(f"{run_dir}/smoke/summary.json"))
    start = smoke["initial"]
    print(f"step 0: loss {start['loss']:.6f} (ln 2 = {math.log(2):.6f}), accuracy {start['reward_accuracy']}, "
          f"margin {start['reward_margin']:.2e}, max |z| {start['max_abs_z']:.2e}")
    if abs(start["loss"] - math.log(2)) > 1e-3 or abs(start["reward_margin"]) > 1e-3:
        raise RuntimeError("policy and reference disagree at step 0: they must start identical")
    if start["reward_accuracy"] > 0:
        print("note: accuracy above 0 at step 0 comes from rounding noise only (see max |z|)")
    if smoke.get("policy_peak_allocated_gib", 0) > MAX_PEAK_GIB:
        raise RuntimeError(f"policy GPU peak {smoke['policy_peak_allocated_gib']} GiB is above {MAX_PEAK_GIB}")
    if smoke["projected_full_run_hours"] > MAX_HOURS_PER_RUN:
        raise RuntimeError(f"projected {smoke['projected_full_run_hours']} h is above {MAX_HOURS_PER_RUN} h")
    if smoke["skipped_steps"] > SMOKE_STEPS // 2:
        raise RuntimeError("most steps were skipped by the loss scaler: fp16 overflow")
    print(f"{name}: timing run within bounds (projected {smoke['projected_full_run_hours']} h), starting the full run")

    run_on_gpus([("train", "0,1", [*common, "--out-dir", f"{run_dir}/run"])], f"{name}_train")
    summary = json.load(open(summary_path))
    assert os.path.exists(f"{run_dir}/run/best/model.safetensors") and os.path.exists(f"{run_dir}/run/final/model.safetensors")
    run([sys.executable, "-m", "pa2.plots", "dpo", "--metrics", f"{run_dir}/run/metrics.jsonl",
         "--out", f"{OUT_DIR}/{name}_curves.png", "--title", title],
        cwd=REPO_DIR, env={**os.environ, "PYTHONPATH": f"{REPO_DIR}/src:{REPO_DIR}"})
    from IPython.display import Image, display
    display(Image(f"{OUT_DIR}/{name}_curves.png"))
    return summary

## Run 1: HH-RLHF

In [ ]:
summaries = {"hh": dpo_run("hh", *RUNS["hh"])}

## Run 2: UltraFeedback

In [ ]:
summaries["ultrafeedback"] = dpo_run("ultrafeedback", *RUNS["ultrafeedback"])

## Both runs side by side

In [ ]:
rows = []
for name, summary in summaries.items():
    for when in ("initial", "best", "final"):
        record = summary[when]
        rows.append((name, when, record.get("step", 0), record["loss"], record["reward_accuracy"],
                     record["reward_margin"], record["chosen_reward"], record["rejected_reward"]))
print(f"{'run':<14}{'point':<9}{'step':>6}{'loss':>9}{'acc':>8}{'margin':>9}{'chosen':>9}{'rejected':>10}")
for name, when, step, loss, acc, margin, chosen, rejected in rows:
    print(f"{name:<14}{when:<9}{step:>6}{loss:>9.4f}{acc:>8.3f}{margin:>9.4f}{chosen:>9.4f}{rejected:>10.4f}")
json.dump(summaries, open(f"{OUT_DIR}/summaries.json", "w"), indent=2)

## Archive the small outputs

In [ ]:
import zipfile


def zip_small_outputs(archive_path, root, skip_dirs=(), skip_suffixes=(".safetensors", ".jsonl.gz")):
    """Zip everything under root except checkpoints and bulky data, so the archive stays small."""
    count = 0
    with zipfile.ZipFile(archive_path, "w", zipfile.ZIP_DEFLATED) as archive:
        for folder, dirs, files in os.walk(root):
            rel_folder = os.path.relpath(folder, root)
            dirs[:] = [d for d in dirs if os.path.normpath(os.path.join(rel_folder, d)) not in skip_dirs]
            for name in sorted(files):
                if name.endswith(skip_suffixes):
                    continue
                path = os.path.join(folder, name)
                archive.write(path, os.path.relpath(path, root))
                count += 1
    print(f"Saved {archive_path} ({count} files, {os.path.getsize(archive_path) / 1e6:.1f} MB)")


zip_small_outputs(f"{WORK_DIR}/task4_dpo_outputs.zip", OUT_DIR,
                  skip_dirs=("hh/run/best", "hh/run/final", "ultrafeedback/run/best", "ultrafeedback/run/final"),
                  skip_suffixes=(".safetensors", "train.jsonl"))